# Recursive agents & budgets

_Compose nested jaz.invoke calls and cap cost across the whole tree._

⏱️ **~20 min** · Part of the [JAZ](https://jaz-lang.ai) tutorial series.

> Run the setup cells once, then work through the rest top to bottom.

## Setup

In [ ]:
# Install JAZ from PyPI
%pip install -q jaz-lang

In [ ]:
import os, getpass

# JAZ routes to providers through LiteLLM and reads each provider's standard
# API-key environment variable. Set at least one. In Colab you can also use the
# secrets manager (key icon in the left sidebar).
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')


In [ ]:
import jaz
from jaz.llm import LiteLLM

# Pick the model all agents use. The model id is a LiteLLM route, 'provider/model'.
# Swap to an Anthropic model (and set ANTHROPIC_API_KEY) if you prefer.
jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))


## Agents that call agents

An agent can call `jaz.invoke` from its own code, spawning nested agents. Recursion is
available by default; cap the depth with a `RecursionLimit` (activated with `with`, so
the cap reaches nested invokes).


In [ ]:
from jaz import invoke
from jaz.hooks import ReturnType, RecursionLimit

data = ["the cat sat", "a dog ran", "birds fly south"]

with RecursionLimit(max_depth=2):
    result = invoke(
        ReturnType(list),
        task="""For each phrase, use jaz.invoke to count its words.
        Return the list of counts.""",
        phrases=data,
    )
result


## Capping cost across the tree

A `BudgetPool` caps total spend and LLM-call count across every nested invoke in its
`with` block.


In [ ]:
from jaz.hooks import BudgetPool

with BudgetPool(cost_budget=0.10, calls_budget=25), RecursionLimit(max_depth=2):  # USD / LLM calls
    result = invoke(
        ReturnType(list),
        task="For each phrase, summarize it in one word using jaz.invoke.",
        phrases=data,
    )
result


## Per-level limits

Recursion depth and per-level iterations are hooks. Activate them with `with` so they
reach nested invokes too, each level counting its own iterations.


In [ ]:
from jaz.hooks import IterationLimit

with RecursionLimit(max_depth=2), IterationLimit(max_iterations=20):
    result = invoke(ReturnType(str), task="A multi-step task")
result


Next: [observability with hooks](https://jaz-lang.ai/docs/tutorials).